In [3]:
%pip install pycocotools ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 8.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [21]:
import random
import sys
import numpy as np
import torch

from pathlib import Path
from pycocotools.coco import COCO
from torch.utils.data import DataLoader

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/ucu-ai-hw-2")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.data_prep import load_annotations
from scripts.data_cv import stratified_group_folds
from scripts.dataset_helpers import DetectionDataset, collate_data
from scripts.model_builders import build_frcnn_model
from scripts.experiment_runner import run_torch_final
from scripts.experiment_results import model_training_results_final

DIRECTORY = PROJECT_ROOT / "data"
INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"
INFO_VAL = DIRECTORY / "val" / "annotations_coco.json"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
EPOCHS = 8
FRAME_STEP = 5
BATCH_SIZE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [28]:
coco_train = COCO(str(INFO_TRAIN))
coco_val = COCO(str(INFO_VAL))

train_frames, train_boxes = load_annotations(coco_train, split="train")
valid_frames, valid_boxes = load_annotations(coco_val, split="val")

train_annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
valid_annotations_by_image = {image_id: group for image_id, group in valid_boxes.groupby("image_id")}
sampled_train_frames = (train_frames.sort_values(["video", "frame_id"]).groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

loading annotations into memory...
Done (t=5.41s)
creating index...
index created!
loading annotations into memory...
Done (t=0.54s)
creating index...
index created!


In [29]:
train_dataset = DetectionDataset(frames=sampled_train_frames, directory=DIRECTORY,
                                 annotations_by_image=train_annotations_by_image,
                                 empty_annotations=train_boxes.iloc[:0].copy(), validation=False)
valid_dataset = DetectionDataset(frames=valid_frames, directory=DIRECTORY,
                                 annotations_by_image=valid_annotations_by_image,
                                 empty_annotations=valid_boxes.iloc[:0].copy(), validation=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collate_data)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, collate_fn=collate_data)

In [30]:
output_root = PROJECT_ROOT / "outputs" / "frcnn_final"
train_kwargs = {
    "train_loader": train_loader,
    "valid_loader": valid_loader,
    "annotations_path": INFO_VAL,
    "device": device,
    "seed": SEED,
    "epochs": EPOCHS,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "lr_scheduler": True,
    "step_size": 4
}

In [31]:
final_history, final_results = run_torch_final(**train_kwargs, model_method=build_frcnn_model, output_dir=output_root)

loading annotations into memory...
Done (t=0.49s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.33s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=78.32s).
Accumulating evaluation results...
DONE (t=3.32s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.154
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.313
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.134
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.059
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.135
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.312
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.099
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.221
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

In [32]:
final_results_wrap = model_training_results_final(output_root)

Training history


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,1.096876,0.00010,0.154391,0.313467,0.058761
1,2,0.912786,0.00010,0.142678,0.312755,0.058224
2,3,0.837375,0.00010,0.166807,0.343651,0.082083
3,4,0.787906,0.00010,0.171285,0.341759,0.061356
4,5,0.670990,0.00001,0.185281,0.362190,0.065708
5,6,0.640327,0.00001,0.188482,0.368054,0.062974
6,7,0.621735,0.00001,0.186629,0.364543,0.064630
7,8,0.607570,0.00001,0.183003,0.360735,0.059951



Best validation results — epoch 6


,mAP,AP50,AP_small
5,0.188482,0.368054,0.062974
